# TERRA Tutorial

- Creators: Sebastian Birk (sb75@sanger.ac.uk), Mohammad Vali Sanian (mohammad.sanian@helsinki.fi)
- Date of Creation: 30.06.2025
- Date of Last Modification: 17.03.2026 (by Sebastian Birk (sb75@sanger.ac.uk)

Run this notebook in the nichejepa environment:  <br>
```source /nfs/users/nfs_s/sb75/.venvs/nichejepa/bin/activate```

To accelerate clustering, run section **2.6** in the rapids-singlecell-nichejepa environment: <br>
```source /etc/profile.d/modules.sh``` <br>
```module load cellgen/conda``` <br>
```conda activate /nfs/team361/sb75/ENVS/rapids-singlecell-nichejepa```

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import gc
import logging
import os
import warnings
from typing import List, Tuple
from tqdm import tqdm

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import scipy.sparse as sp
try:
    import squidpy as sq
except:
    print("Could not import squidpy")
    import rapids_singlecell as rsc
import torch
from datasets import concatenate_datasets, load_from_disk
from matplotlib.colors import Normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from app.inference import (embed_dataset,
                           harmonize_adata,
                           perturb_dataset,
                           tokenize_adata,
                           harmonize_tokenize_embed_pipeline,
                           get_gene_embed,
                           get_average_gene_embed,
                           get_spatial_score,
                           get_emd_distance)
from app.utils import init_model, init_opt, load_checkpoint
from nichejepa.utils.evaluation import (get_top_gene_score,
                                        get_top_gene_pairs)

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

**TODO**: Specify timestamp of model checkpoint.

In [ ]:
model_timestamp = '01112025_081731_614' # single tissue model
model_timestamp = '11012026_190057_836' # cross-tissue model
#model_timestamp = '27022026_083110_268' # mmb0-1b_smb1-1b_1p mouse model 64 sequence length
#model_timestamp = '04032026_215925_595' # mmb0-1b_smb1-1b_2p mouse model
#model_timestamp = '03032026_094533_276' # mmb0-1b_smb1-1b_2p mouse model
#model_timestamp = '27022026_083110_268' # test model

**TODO**: Adjust `artifacts_folder_path` to a location where you want store outputs.

In [ ]:
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'

In [ ]:
model_folder_path = f'{artifacts_folder_path}/models/{model_timestamp}'
results_folder_path = f'{artifacts_folder_path}/results/{model_timestamp}'
cache_directory_path = f'{artifacts_folder_path}/cache'
tokenized_data_folder_path = f'{artifacts_folder_path}/tokenized_data'
embedded_data_folder_path = f'{artifacts_folder_path}/embedded_data'

### 1.3 Define Config

**TODO**: To provide your input data, you have three different options. Please pick the correct option for your case and set the config in the following notebook cell accordingly.
- Option 1: specify a single file (`file_path`) containing an AnnData object with a single section.
    - In this case, set `folder_path` and `sample_key` to None.
- Option 2: specify a folder (`folder_path`) with multiple AnnData objects (one per section).
    - In this case, set `file_path` and `sample_key` to None.
- Option 3: specify a file (`file_path`) containing an AnnData object with multiple sections.
    - In this case, set `folder_path` to None and specify the `sample_key` as the adata.obs column that is unique for each section.
 
For all options, also define a project name where outputs will be saved.

In [ ]:
### Option 1 examples ###
#project_name = 'kidney_disease_integration_cosmx_test'
#file_path = '/nfs/team361/dj17/TERRA_DJ/Integration_disease/adatas_processed/CosMx_Biopsy3_commongenes_qc.h5ad'
#sample_key = None
#folder_path = None

#project_name = 'kidney_healthy_integration_slideseqv2_test'
#file_path = '/nfs/team361/dj17/TERRA_DJ/Integration_healthy/adatas_processed/Puck_200104_15_with_emb_NEMO_xeniumCOMMON_qc.h5ad'
#sample_key = None
#folder_path = None

#project_name = 'kidney_healthy_integration_xenium_test'
#file_path = '/nfs/team361/dj17/TERRA_DJ/Integration_healthy/adatas_processed/CV9-KID-0-FT-5_with_emb_NEMO_xeniumCOMMON_qc.h5ad'
#sample_key = None
#folder_path = None

#project_name = 'skin_eczema'
#file_path = '/nfs/team361/sb75/DATASETS/silver/xhs1000-39b_1p/adata_batch19.h5ad'
#sample_key = None
#folder_path = None

#project_name = 'whole_embryo'
#file_path = '/lustre/scratch126/cellgen/lotfollahi/jr43/projects/xenium_to_anndata/outputs/SOB26-WEM-FFPE-1-S20__20240912__123924.h5ad'
#sample_key = None
#folder_path = None

### Option 2 examples ###
#project_name = 'kidney_disease_integration'
#folder_path = '/nfs/team361/dj17/TERRA_DJ/Integration_disease/adatas_processed'
#sample_key = None
#file_path =  None

project_name = 'kidney_disease_integration_6k'
folder_path = '/nfs/team361/dj17/TERRA_DJ/Integration_disease/adatas/6kintersection'
sample_key = None
file_path =  None

#project_name = 'mmb0-1b_smb1-1b_1p'
#folder_path = '/nfs/team361/sb75/DATASETS/silver/mmb0-1b_smb1-1b_1p'
#sample_key = None
#file_path =  None

#project_name = 'kidney_healthy_integration'
#folder_path = '/nfs/team361/dj17/TERRA_DJ/Integration_healthy/adatas_processed'
#sample_key = None
#file_path =  None

#project_name = 'kidney_disease_integration_edinburgh_cosmx_1k'
#folder_path = '/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/adatafiles_zeroshot/EdinburghCosMx_1k_h5ads'
#sample_key = None
#file_path = None

#project_name = 'kidney_disease_integration_susztak_cosmx_1k'
#folder_path = '/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/adatafiles_zeroshot/SusztakCosMx_h5ads'
#sample_key = None
#file_path = None

#project_name = 'kidney_disease_integration_ln_cosmx_1k'
#folder_path = '/nfs/team361/dj17/NEMO_DJ/KidneyToxicity/adatafiles_zeroshot/LNCosMx_h5ads'
#sample_key = None
#file_path = None

### Option 3 examples ###
#project_name = 'fetal_pancreas'
#file_path = '/lustre/scratch126/cellgen/lotfollahi/so21/pancreas/terra/pancreas_noStomach.h5ad'
#sample_key = 'Sample'
#folder_path = None

#project_name = 'adult_pancreas'
#file_path = '/lustre/scratch126/cellgen/lotfollahi/ms83/pancreas/nPOD-T1D/adata_healthy_pancreas_DJ1_DJ3.h5ad'
#sample_key = 'slide_ID'
#folder_path = None

**TODO**: Specify other dataset-specific parameters.

In [ ]:
label_keys = ['total_counts'] # 'nichecompass_latent_cluster', 'total_counts', 'n_genes_by_counts'
spot_size = 100

**TODO**: Specify inference and analysis parameters.

In [ ]:
# Harmonization params
min_cells_per_gene = 0
min_genes_per_cell = 10

# Tokenization params
nproc = 4
processing_mode = 'parallel'
use_generator = True
num_shards = 32
ensembl_release = 110
species = 'mouse' # human'
gene_mapping_dict_file_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl' # None, '/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl'
gene_occurrence_count_file_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_occurence_count_dict.pkl' # None, '/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_occurence_count_dict.pkl'
gene_occurrence_count_filter_value = 10

# Model params
emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_genes = None
top_k = None

# Data params
batch_key = 'batch'
n_pca_pcs = 20

# Clustering params
n_pcs = None
latent_leiden_resolution = 1.0

### 1.4 Run Notebook Setup

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
# Set plotting parameters
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

sc.settings.figdir = f'{results_folder_path}/{project_name}'
sc.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
# Create folders
os.makedirs(results_folder_path, exist_ok=True)
os.makedirs(f'{results_folder_path}/{project_name}', exist_ok=True)
os.makedirs(tokenized_data_folder_path, exist_ok=True)
os.makedirs(embedded_data_folder_path, exist_ok=True)
os.makedirs(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}", exist_ok=True)

### 1.5 Define Functions

In [ ]:
def subset_by_cell_ids(dataset, cell_id_list):
    """
    Takes a huggingface dataset and a list of cell ids and returns the corresponding filtered dataset object.
    """
    cell_id_set = set(cell_id_list)
    cell_ids = dataset['cell_id']
    indices = [i for i, cid in tqdm(enumerate(cell_ids), total=len(cell_ids), desc="Finding matching indices") if cid in cell_id_set]
    return dataset.select(indices)

In [ ]:
def summarize_gene_expression(
    adata,
    subset_cell_ids: list,
    top_n: int = 10,
    ensembl_col: str = "ensembl_id"
):
    """
    Filter AnnData to a subset of cells, compute gene expression summaries,
    and return top expressed genes with Ensembl IDs.

    Args:
        adata (AnnData): Pre-loaded AnnData object.
        subset_cell_ids (list): List of cell IDs to filter.
        top_n (int): Number of top expressed genes to return.
        ensembl_col (str): Column in adata.var that contains Ensembl IDs.

    Returns:
        combined_counts (pd.DataFrame): Filtered cell-by-gene count matrix.
        top_gene_df (pd.DataFrame): DataFrame with top gene names and Ensembl IDs.
    """
    if 'cell_id' not in adata.obs.columns:
        raise KeyError("Expected `adata.obs` to contain a 'cell_id' column.")
    if ensembl_col not in adata.var.columns:
        raise KeyError(f"Expected `adata.var` to contain '{ensembl_col}' column.")

    # Filter to subset of cells
    mask = adata.obs['cell_id'].astype(str).isin(set(subset_cell_ids))
    filtered = adata[mask].copy()
    if filtered.n_obs == 0:
        raise ValueError("No matching cells found in the provided subset.")

    # Extract dense matrix
    matrix = filtered.X.toarray() if hasattr(filtered.X, "toarray") else filtered.X
    combined_counts = pd.DataFrame(matrix, columns=filtered.var_names, index=filtered.obs_names)

    # Compute mean expression and get top N genes
    mean_expr = combined_counts.mean(axis=0)
    top_genes = mean_expr.nlargest(top_n)

    # Map to Ensembl IDs
    var_df = adata.var.loc[top_genes.index]
    top_gene_df = pd.DataFrame({
        "gene_name": top_genes.index,
        "ensembl_id": var_df[ensembl_col].values,
        "mean_expression": top_genes.values
    })

    return combined_counts, top_gene_df

## 2. Analysis

### 2.1 (Optional) Visualize Data

In [ ]:
if folder_path: # Option 2
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".h5ad"):
            file_path = os.path.join(folder_path, file_name)
            print(f"Reading {file_path}.")
    
            adata = sc.read_h5ad(file_path)
            print(adata)
            if 'spatial' in adata.uns.keys():
                del(adata.uns['spatial'])

            for label_key in label_keys:
                sc.pl.spatial(adata,
                              spot_size=spot_size,
                              color=label_key)
else:
    adata = sc.read_h5ad(file_path)
    print(adata)
    if 'spatial' in adata.uns.keys():
        del(adata.uns['spatial'])

    if sample_key: # Option 3
        for batch in adata.obs[sample_key].unique().tolist():
            adata_batch = adata[adata.obs[sample_key] == batch]
            for label_key in label_keys:
                sc.pl.spatial(adata_batch,
                              spot_size=spot_size,
                              color=label_key) 
    else: # Option 1
        for label_key in label_keys:
            sc.pl.spatial(adata,
                          spot_size=spot_size,
                          color=label_key) 

### 2.2 Harmonize, Tokenize & Embed Data to Retrieve Global Embeddings

**TODO**: To retrieve TERRA embeddings, the input AnnData object first needs to be harmonized and tokenized. You can either run the end-to-end pipeline from step **2.2.1** OR the step-by-step approach from **2.2.2**.

#### 2.2.1 End-to-end Pipeline

**TODO**: Skip this if you want to run the step-by-step-approach in **2.2.2**.

In [ ]:
if folder_path:
    adata_list = []
    for i, file_name in enumerate(os.listdir(folder_path)):
        if file_name.endswith(".h5ad") or file_name.endswith(".h5ad.gz"):
            file_path = os.path.join(folder_path, file_name)
            sample_name = file_name.replace(".h5ad.gz", ".h5ad")
            sample_name = sample_name.replace(".h5ad", "")
    
            adata_sample = sc.read_h5ad(file_path)

            adata_sample = harmonize_tokenize_embed_pipeline(
                adata=adata_sample,
                sample_key=sample_key,
                model_folder_path=model_folder_path,
                cache_directory_path=cache_directory_path,
                gene_mapping_dict_file_path=gene_mapping_dict_file_path,
                gene_occurrence_count_file_path=gene_occurrence_count_file_path,
                gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
                ensembl_release=ensembl_release,
                species=species,
                gene_perturb_df=None,               
                nproc=nproc,
                processing_mode=processing_mode,
                save_dataset_path=f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                num_shards=num_shards,
                emb_layer=emb_layer,
                agg_excluded_genes=agg_excluded_genes,
                top_k=top_k,
                batch_size=batch_size,
                pin_memory=pin_memory,
                num_workers=num_workers,
                use_generator=use_generator,
                add_neigh_cell_ids=False,
                min_cells_per_gene=min_cells_per_gene,
                min_genes_per_cell=min_genes_per_cell,
                ignore_spc_tokens=False)

            adata_sample.obs[batch_key] = sample_name
            adata_sample.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
            adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
else:
    file_name = file_path.split('/')[-1]
    sample_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = sample_name.replace(".h5ad", "")

    adata = sc.read_h5ad(file_path)
    
    adata = harmonize_tokenize_embed_pipeline(
        adata=adata,
        sample_key=sample_key,
        model_folder_path=model_folder_path,
        cache_directory_path=cache_directory_path,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
        ensembl_release=ensembl_release,
        species=species,
        gene_perturb_df=None,               
        nproc=nproc,
        processing_mode=processing_mode,
        save_dataset_path=f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
        num_shards=num_shards,
        emb_layer=emb_layer,
        agg_excluded_genes=agg_excluded_genes,
        top_k=top_k,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        use_generator=use_generator,
        add_neigh_cell_ids=False,
        min_cells_per_gene=min_cells_per_gene,
        min_genes_per_cell=min_genes_per_cell,
        ignore_spc_tokens=True)

    adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")

#### 2.2.2 Step-by-Step Approach

**TODO**: Skip this if you ran the end-to-end pipeline in **2.2.1**.

##### 2.2.2.1 Harmonize Data

In [ ]:
if folder_path:
    adata_list = []
    for i, file_name in enumerate(os.listdir(folder_path)):
        if file_name.endswith(".h5ad") or file_name.endswith(".h5ad.gz"):
            file_path = os.path.join(folder_path, file_name)
            sample_name = file_name.replace(".h5ad.gz", ".h5ad")
            sample_name = sample_name.replace(".h5ad", "")
            
            adata_sample = sc.read_h5ad(file_path)

            adata_sample.obs_names_make_unique()

            adata_sample = harmonize_adata(
                       adata_sample,
                       gene_mapping_dict_file_path=gene_mapping_dict_file_path,
                       gene_occurrence_count_file_path=gene_occurrence_count_file_path,
                       gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
                       ensembl_release=ensembl_release,
                       species=species,
                       min_cells_per_gene=min_cells_per_gene,
                       min_genes_per_cell=min_genes_per_cell)
            adata_sample.obs[batch_key] = sample_name
            adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
elif sample_key:
    adata = sc.read_h5ad(file_path)
    adata_list = []
    samples = adata.obs[sample_key].unique().tolist()
    for sample_name in samples:
        adata_sample = adata[adata.obs[sample_key] == sample]

        adata_sample.obs_names_make_unique()

        adata_sample = harmonize_adata(
            adata_sample,
            gene_mapping_dict_file_path=gene_mapping_dict_file_path,
            gene_occurrence_count_file_path=gene_occurrence_count_file_path,
            gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
            ensembl_release=ensembl_release,
            species=species,
            min_cells_per_gene=min_cells_per_gene,
            min_genes_per_cell=min_genes_per_cell)
        adata_sample.obs[batch_key] = sample_name
        adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
else:
    adata = sc.read_h5ad(file_path)
    adata = harmonize_adata(
        adata,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
        ensembl_release=ensembl_release,
        species=species,
        min_cells_per_gene=min_cells_per_gene,
        min_genes_per_cell=min_genes_per_cell)
print(adata)

##### 2.2.2.2 Tokenize Data

Run **2.2.2.1 Harmonize Data** before this step.

In [ ]:
if folder_path or sample_key:
    dataset_list = []
    for adata_sample in adata_list:
        dataset_sample = tokenize_adata(
            adata_sample,
            model_folder_path,
            cache_directory_path,
            nproc=nproc,
            processing_mode=processing_mode)
        item = dataset_sample[0]
        print(item)
        dataset_list.append(dataset_sample)
else:
    dataset = tokenize_adata(
        adata=adata,
        model_folder_path=model_folder_path,
        cache_directory_path=cache_directory_path,             
        nproc=nproc,
        processing_mode=processing_mode)
    item = dataset[0]
    print(item)

In [ ]:
# Save Huggingface datasets for each sample
if folder_path or sample_key:
    for i, sample_name in enumerate(adata.obs[batch_key].unique().tolist()):
        dataset = dataset_list[i]
        dataset.save_to_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                             num_shards=num_shards)
else:
    file_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = file_name.replace(".h5ad", "")
    dataset.save_to_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                         num_shards=num_shards)

##### 2.2.2.3 Load & Concatenate Tokenized Data

In [ ]:
# Load Huggingface datasets for each sample and concatenate
if folder_path or sample_key:
    dataset_list = []
    for i, sample_name in enumerate(adata.obs[batch_key].unique().tolist()):
        sample_dataset = load_from_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset")
        dataset_list.append(sample_dataset)
else:
    file_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = file_name.replace(".h5ad", "")
    dataset = load_from_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset")

##### 2.2.2.4 Retrieve Embeddings

In [ ]:
if folder_path:
    for adata_sample, dataset_sample in zip(adata_list, dataset_list):

        sample_name = adata_sample.obs[batch_key].unique().tolist()[0]
        
        output_embed = embed_dataset(
            dataset=dataset_sample,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_genes=agg_excluded_genes,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers,
            ignore_spc_tokens=True)

        # Add embeddings to adata
        for key, values in output_embed.items():
            adata_sample.obsm[key] = values

        adata_sample.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
elif sample_key:
    adata_emb_list = []
    for adata_sample, dataset_sample in zip(adata_list, dataset_list):

        sample_name = adata_sample.obs[batch_key].unique().tolist()[0]
        
        output_embed = embed_dataset(
            dataset=dataset_sample,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_genes=agg_excluded_genes,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers,
            ignore_spc_tokens=True)

        # Add embeddings to adata
        for key, values in output_embed.items():
            adata_sample.obsm[key] = values

        adata_emb_list.append(adata_sample)
        adata = ad.concat(adata_emb_list)
        file_name = file_path.split('/')[-1]
        adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{file_name}")    
else:
    file_name = file_path.split('/')[-1]
    sample_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = sample_name.replace(".h5ad", "")
    
    output_embed = embed_dataset(
        dataset=dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        agg_excluded_genes=agg_excluded_genes,
        top_k=top_k,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        ignore_spc_tokens=True)
    
    # Add embeddings to adata
    for key, values in output_embed.items():
        adata.obsm[key] = values

    adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")

### 2.3 (Optional) Retrieve Average Gene Embeddings
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
list(adata.var['ensembl_id'])

In [ ]:
cell_gene_ensembl_id = list(top_gene_df['ensembl_id'])
neighborhood_gene_ensembl_id = list(top_gene_df['ensembl_id']) # ['all'] * len(perturbed_cell_id)

cell_gene_ensembl_id = list(adata.var['ensembl_id'])
neighborhood_gene_ensembl_id = list(adata.var['ensembl_id'])

In [ ]:
output_average_gene_embed = get_average_gene_embed(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
adata.uns['cell_gene_emb_average_per_data'] = output_average_gene_embed['cell_gene_emb_average_per_data']
adata.uns['cell_gene_emb_counts_per_data'] = output_average_gene_embed['cell_gene_emb_counts_per_data']
adata.uns['neighborhood_gene_emb_average_per_data'] = output_average_gene_embed['neighborhood_gene_emb_average_per_data']
adata.uns['neighborhood_gene_emb_counts_per_data'] = output_average_gene_embed['neighborhood_gene_emb_counts_per_data']

In [ ]:
adata.uns['cell_gene_emb_average_per_data']

In [ ]:
# 1) You can choose any of the embeddings, which could be either a cell or a neighborhood.
gene_emb_name = 'neighborhood_gene_emb_average_per_data'
gene_emb_name = 'cell_gene_emb_average_per_data'


In [ ]:
gene_emb_name

In [ ]:
gene_emb = np.array(adata.uns[gene_emb_name])  # (n_genes, emb_dim)
gene_ids =  np.array(cell_gene_ensembl_id,  dtype=str)                  # (n_genes,)

# 2) Remove any gene rows containing NaNs
mask = ~np.isnan(gene_emb).any(axis=1)
gene_emb_clean = gene_emb[mask]
gene_ids_clean = gene_ids[mask]

# 3) Build a small AnnData where each "obs" is one gene
adata_genes = ad.AnnData(
    X=gene_emb_clean,
    obs=pd.DataFrame({'gene_id': gene_ids_clean})
)
adata_genes.obs_names = gene_ids_clean.astype(str)

# 4) Compute a neighborhood graph on the embeddings
sc.pp.neighbors(
    adata_genes,
    use_rep='X',
    n_neighbors=15,
    metric='euclidean'
)

# 5) Run UMAP
sc.tl.umap(
    adata_genes,
    #min_dist=0.5,
    #spread=1.0
)


# 7) (Optional) Quick plot
sc.pl.umap(
    adata_genes,
    title='Gene embedding UMAP',
    show=True
)


In [ ]:
sc.tl.pca(adata_genes, n_comps=2, use_highly_variable=False, svd_solver='arpack')

sc.pl.pca(
    adata_genes,
    components=['1,2'],   # plot PC1 vs PC2
    color=None,
    title='Gene embeddings PCA (PC1 vs PC2)',
    show=True
)


In [ ]:
# Store results here
results = []

# Define parameters
knn_values = [5, 10, 15, 20]
resolutions = np.arange(0.1, 1.1, 0.1)  # 0.1 to 1.0

# Loop over each KNN and resolution combination
for knn in knn_values:
    sc.pp.neighbors(adata_genes, n_neighbors=knn, use_rep='X')  # use 'X' or other embedding like 'X_pca'
    
    for res in resolutions:
        # Run Leiden clustering
        sc.tl.leiden(adata_genes, resolution=res, key_added='leiden_tmp')
        
        # Compute silhouette score
        labels = adata_genes.obs['leiden_tmp']
        if len(np.unique(labels)) < 2:
            score = np.nan  # silhouette score not defined for 1 cluster
        else:
            score = silhouette_score(adata_genes.obsm['X_pca'], labels)  # you can change to 'X' if no PCA
        
        # Store results
        results.append({
            'knn': knn,
            'resolution': round(res, 2),
            'silhouette_score': score
        })

# Convert results to DataFrame
df = pd.DataFrame(results)
df['combo'] = df['knn'].astype(str) + '_k_' + df['resolution'].astype(str)

# Plot
plt.figure(figsize=(12, 6))
df_sorted = df.sort_values('silhouette_score', ascending=False)
plt.bar(df_sorted['combo'], df_sorted['silhouette_score'], color='skyblue')
plt.xticks(rotation=90)
plt.ylabel("Silhouette Score")
plt.xlabel("KNN_Resolution Combination")
plt.title("Silhouette Score for Different KNN and Leiden Resolution Settings")
plt.tight_layout()
plt.show()


In [ ]:
sc.pp.neighbors(adata_genes,n_neighbors=20)
sc.tl.leiden(adata_genes,resolution=0.3,key_added='r9')

In [ ]:
sc.pl.umap(adata_genes,color='r9',frameon=False)

In [ ]:
id_to_name = {ensg: name for name, ensg in adata.var['ensembl_id'].items()}

In [ ]:
interesting_group = '10'
ens_interesting_group = list(adata_genes.obs[adata_genes.obs['r9']==interesting_group]['gene_id'])

In [ ]:
genes  = [ id_to_name.get(ensg, "UNKNOWN") for ensg in ens_interesting_group]

In [ ]:
len(genes)

In [ ]:
# 2) Your input gene list
# 3) Call gseapy.enrichr
#    You can choose any Enrichr library; here are some examples:
# I should create a mapping from gene_id to ens
# from 110 release
libraries = [
    'KEGG_2019_Human',
    'GO_Biological_Process_2021',
    'Reactome_Pathways_2024',
    'MSigDB_Hallmark_2020'
]

# Run ORA for each library
results = {}
for lib in libraries:
    enr = gp.enrichr(
        gene_list = genes,
        gene_sets  = lib,
        outdir     = None,        # don't write files
        cutoff     = 0.05,         # only show terms with adjusted p < 0.05
    )
    # enr.results is a pandas DataFrame
    results[lib] = enr.results

# 4) Inspect top enriched terms for each library
for lib, df in results.items():
    print(f"\n=== Top 5 terms in {lib} ===")
    display(df[['Term','Adjusted P-value','Combined Score']].head(20))


### 2.4 (Optional) Retrieve Cell-level Gene Embeddings

In [ ]:
adata.var

In [ ]:
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

**TODO**: Specify the gene ensembl IDs for which you want to retrieve cell-level gene embeddings. 

In [ ]:
cell_gene_ensembl_id = ['ENSG00000104419', 'ENSG00000130402']
neighborhood_gene_ensembl_id = ['ENSG00000104419','ENSG00000130402'] # ['all'] * len(perturbed_cell_id)

In [ ]:
output_gene_embed = get_gene_embed( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
for gene_ens in output_gene_embed.keys():
    adata.obsm[gene_ens] = output_gene_embed[gene_ens]

In [ ]:
output_gene_embed

In [ ]:
A = adata.obsm['neighborhood_emb_geneENSG00000104419']
B = adata.obsm['neighborhood_emb_geneENSG00000130402']

# Compute row-wise norms
norm_A = np.linalg.norm(A, axis=1)
norm_B = np.linalg.norm(B, axis=1)

# Mask: keep only rows where BOTH are non-zero
mask = (norm_A > 0) & (norm_B > 0)

# Compute cosine similarity only for valid rows
cos_sim = np.sum(A[mask] * B[mask], axis=1) / (norm_A[mask] * norm_B[mask])

print(cos_sim)

In [ ]:
def compute_mean_unmasked_emb(emb: torch.Tensor,
                              mask: torch.Tensor,
                              ) -> torch.Tensor:
    """
    Compute the mean of unmasked embedding positions.
    
    Parameters
    -----------
    emb:
        The input embeddings tensor (3D).
    mask:
        A 2D boolean mask tensor indicating the sequence positions that mean
        should be computed over.
    
    Returns
    -----------
    mean_emb:
        The mean embedding tensor.

    Raises
    -----------
    ValueError: If the emb tensor is not 3D.
    """
    # Use broadcasting to sum embeddings across unmasked positions
    if emb.dim() == 3:
        # If the embeddings tensor has 3 dimensions (batch_size,
        # sequence_length, embedding_dim), broadcast the mask to match the
        # dimensions of emb. The mask tensor is initially (batch_size,
        # sequence_length), so we unsqueeze to (batch_size, sequence_length, 1)
        masked_emb = emb * mask.unsqueeze(2) # broadcast the mask along the
                                             # embedding dimension

        # Sum the masked embeddings along the sequence dimension
        sum_emb = masked_emb.sum(1)

        # Calculate the mean by dividing the summed embeddings by the number of
        # unmasked positions. The mask is summed to count unmasked tokens, and
        # view(-1, 1) ensures the resulting tensor has the correct dimensions
        # for broadcasting during division. The + 1e-9 will handle the case
        # where we are retrieving a gene that may have
        # mask.sum(dim).view(-1, 1).float() = 0
        mean_emb = sum_emb / (mask.sum(1).view(-1, 1).float() + 1e-9)

    else:
        raise ValueError('Expected a 3D tensor for emb, but got a tensor with'
                         f'{emb.dim()} dimensions.')

    return mean_emb

In [ ]:
A[mask]

In [ ]:
B[mask]

In [ ]:
mask

In [ ]:
cos_sim.shape

In [ ]:
np.mean(cos_sim)

### 2.5 (Optional) Analyze Raw Counts

#### 2.5.1 Visualize PCA-based UMAP baseline

In [ ]:
# Create dataset-specific label columns
if project_name == 'kidney_healthy_integration':
    adata.obs['assay'] = adata.obs['batch'].str.startswith(('DI', 'CV'), na=False) \
        .map({True: 'xenium', False: 'slideseqv2'})
    
    adata.obs['source'] = np.select(
        [
            adata.obs['batch'].str.startswith('DI', na=False),
            adata.obs['batch'].str.startswith('CV', na=False)
        ],
        ['DI', 'CV'],
        default='Puck'
    )

    label_keys = ['assay', 'source']

In [ ]:
# Plot cell-level expression-based UMAP
try: # if rapids_singlecell is installed
    rsc.pp.pca(adata,
               n_comps=n_pca_pcs)
    
    rsc.pp.neighbors(adata,
                     n_neighbors=15,
                     use_rep="X_pca",
                     n_pcs=None,
                     key_added='expr')
    
    rsc.tl.umap(adata,
               neighbors_key='expr',
               key_added=f'expr_umap')
except NameError:
    sc.pp.pca(adata,
              n_comps=n_pca_pcs)
    
    sc.pp.neighbors(adata,
                    n_neighbors=15,
                    use_rep="X_pca",
                    n_pcs=None,
                    key_added='expr')
    
    sc.tl.umap(adata,
               neighbors_key='expr',
               key_added=f'expr_umap')
    
adata.obsm['X_umap'] = adata.obsm[f'expr_umap']

n_pca_pcs_string = f"_{n_pca_pcs}_pcs"
sc.pl.umap(
    adata,
    color=batch_key,
    show=False
)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_umap_by_{batch_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

for label_key in label_keys:
    sc.pl.umap(
        adata,
        color=label_key,
        show=False
    )
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_umap_by_{label_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Plot neighborhood-level expression-based UMAP
adata.layers['X_neigh'] = (
    adata.obsp['spatial_connectivities'].T @ adata.X)

try: # if rapids_singlecell is installed
    rsc.pp.pca(adata,
               layer="X_neigh",
               n_comps=n_pca_pcs)

    rsc.pp.neighbors(adata,
        n_neighbors=15,
        use_rep="X_pca",
        n_pcs=None,
        key_added="expr_neigh")

    rsc.tl.umap(adata,
                neighbors_key="expr_neigh",
                key_added="expr_neigh_umap")

except NameError:
    sc.pp.pca(adata,
              layer="X_neigh",
              n_comps=n_pca_pcs)

    sc.pp.neighbors(adata,
                    n_neighbors=15,
                    use_rep="X_pca",
                    n_pcs=None,
                    key_added="expr_neigh")

    sc.tl.umap(adata,
               neighbors_key="expr_neigh",
               key_added="expr_neigh_umap")


adata.obsm["X_umap"] = adata.obsm["expr_neigh_umap"]

n_pca_pcs_string = f"_{n_pca_pcs}_pcs"
sc.pl.umap(adata, color=batch_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_neigh_umap_by_{batch_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

for label_key in label_keys:
    sc.pl.umap(adata, color=label_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_neigh_umap_by_{label_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

torch.cuda.empty_cache()
gc.collect()

#### 2.5.2 Visualize PCA-based Clustering

In [ ]:
# Plot cell-level expression-based clusters
latent_cluster_key = f"expr_clusters_res_{latent_leiden_resolution}"

try:
    rsc.tl.leiden(adata,
                  neighbors_key='expr',
                  key_added=latent_cluster_key,
                  resolution=latent_leiden_resolution,
                  flavor="igraph",
                  n_iterations=2)
except NameError:
    sc.tl.leiden(adata,
                 neighbors_key='expr',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution,
                 flavor="igraph",
                 n_iterations=2)

adata.obsm["X_umap"] = adata.obsm["expr_umap"]

sc.pl.umap(adata, color=latent_cluster_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"expr_umap_by_{sample_key}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

samples = adata.obs[sample_key].unique().tolist()
for i, sample in enumerate(samples):
    adata_sample = adata[adata.obs[sample_key] == sample]
    sc.pl.spatial(adata_sample,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_size,
                  save=f'tissue_by_{latent_cluster_key}.png')
    os.rename(f'{results_folder_path}/showtissue_by_{latent_cluster_key}.png',
              f'{results_folder_path}/tissue_{sample}_by_{latent_cluster_key}.png')

In [ ]:
# Plot neighborhood-level expression-based clusters
latent_cluster_key = f"expr_neigh_clusters_res_{latent_leiden_resolution}"

try:
    rsc.tl.leiden(adata,
                  neighbors_key='expr_neigh',
                  key_added=latent_cluster_key,
                  resolution=latent_leiden_resolution,
                  flavor="igraph",
                  n_iterations=2)
except NameError:
    sc.tl.leiden(adata,
                 neighbors_key='expr_neigh',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution,
                 flavor="igraph",
                 n_iterations=2)

adata.obsm["X_umap"] = adata.obsm["expr_neigh_umap"]

sc.pl.umap(adata, color=latent_cluster_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"expr_neigh_umap_by_{sample_key}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

samples = adata.obs[sample_key].unique().tolist()
for i, sample in enumerate(samples):
    adata_sample = adata[adata.obs[sample_key] == sample]
    sc.pl.spatial(adata_sample,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_size,
                  save=f'tissue_by_{latent_cluster_key}.png')
    os.rename(f'{results_folder_path}/showtissue_by_{latent_cluster_key}.png',
              f'{results_folder_path}/tissue_{sample}_by_{latent_cluster_key}.png')

#### 2.5.3 Summarize Gene Expression

In [ ]:
combined_counts, top_gene_df = summarize_gene_expression(adata, list(adata.obs['cell_id']),top_n=1000)
display(combined_counts)
display(top_gene_df)

### 2.6 Analyze Embeddings

#### 2.6.1 Load Embeddings

In [ ]:
# Load AnnData with embeddings
if folder_path:
    files = [os.path.join(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}", f) for f in os.listdir(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}") if f.endswith(".h5ad")]
    adata_list = []
    for f in files:
        print(f"Reading {f}")
        adata_file = ad.read_h5ad(f)
        adata_list.append(adata_file)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
    for key in adata_list[0].obsp.keys():
        blocks = [a.obsp[key] for a in adata_list]
        adata.obsp[key] = sp.block_diag(blocks, format="csr")
else:
    file_name = file_path.split('/')[-1]
    adata = sc.read_h5ad(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{file_name}")
print(adata)

#### 2.6.2 Validate & Visualize Embeddings

In [ ]:
# Compute average cosine similarity to check embeddings are not collapsed
# (this should be < 0.9 otherwise the embeddings might not be useful; this can happen if the gene panel is too small)
for emb_key in ['neighborhood_emb', 'cell_emb']:
    X = adata.obsm[emb_key]
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    n = X.shape[0]
    S = X.sum(axis=0)
    avg_cos_sim = (S @ S - n) / (n * (n - 1))
    print(f'{emb_key} average cosine similarity: {avg_cos_sim}')

In [ ]:
# Create dataset-specific label columns
if project_name == 'kidney_healthy_integration'
    adata.obs['assay'] = adata.obs['batch'].str.startswith(('DI', 'CV'), na=False) \
        .map({True: 'xenium', False: 'slideseqv2'})
    
    adata.obs['source'] = np.select(
        [
            adata.obs['batch'].str.startswith('DI', na=False),
            adata.obs['batch'].str.startswith('CV', na=False)
        ],
        ['DI', 'CV'],
        default='Puck'
    )

    label_keys = ['assay', 'source']
elif project_name == 'kidney_disease_integration_6k'
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

    label_keys = ['assay']
elif project_name == 'kidney_disease_integration'
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

In [ ]:
# Plot embedding-based UMAPs
for emb_key in ['neighborhood_emb', 'cell_emb']:
    try: # if rapids_singlecell is installed
        rsc.pp.neighbors(adata,
                         n_neighbors=15,
                         use_rep=emb_key,
                         n_pcs=n_pcs,
                         key_added=emb_key)
        
        rsc.tl.umap(adata,
                    neighbors_key=emb_key,
                    key_added=f'{emb_key}_umap')
    except NameError:
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=emb_key,
                        n_pcs=n_pcs,
                        key_added=emb_key)
        
        sc.tl.umap(adata,
                   neighbors_key=emb_key,
                   key_added=f'{emb_key}_umap')
    
    adata.obsm["X_umap"] = adata.obsm[f"{emb_key}_umap"]

    n_pcs_string = f"_{n_pcs}_pcs"
    sc.pl.umap(adata, color=batch_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/{emb_key}_umap_by_{batch_key}{n_pcs_string if n_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    for label_key in label_keys:
        sc.pl.umap(adata, color=label_key, show=False)
        plt.savefig(os.path.join(results_folder_path, f"{project_name}/{emb_key}_umap_by_{label_key}{n_pcs_string if n_pcs else ''}.png"), dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

In [ ]:
# Save intermediary results
adata.write(os.path.join(results_folder_path, f"{project_name}/adata_processed.h5ad"))

#### 2.6.3 Niche & Cell Type Identification

In [ ]:
# Plot embedding-based clusters
for emb_key in ['neighborhood_emb', 'cell_emb']:
    latent_cluster_key = f"{emb_key}_clusters_res_{latent_leiden_resolution}"
    
    try:
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=latent_cluster_key,
                      resolution=latent_leiden_resolution,
                      flavor="igraph",
                      n_iterations=2)
    except NameError:
        sc.tl.leiden(adata,
                     neighbors_key=emb_key,
                     key_added=latent_cluster_key,
                     resolution=latent_leiden_resolution,
                     flavor="igraph",
                     n_iterations=2)
    
    adata.obsm["X_umap"] = adata.obsm[f"{emb_key}_umap"]
    
    sc.pl.umap(adata, color=latent_cluster_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/{emb_key}_umap_by_{latent_cluster_key}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    
    samples = adata.obs[batch_key].unique().tolist()
    for i, sample in enumerate(samples):
        adata_sample = adata[adata.obs[batch_key] == sample]
        sc.pl.spatial(adata_sample,
                      color=latent_cluster_key,
                      palette=custom_palette,
                      spot_size=spot_size,
                      save=f'tissue_by_{latent_cluster_key}.png')
        os.rename(f'{results_folder_path}/{project_name}/showtissue_by_{latent_cluster_key}.png',
                  f'{results_folder_path}/{project_name}/tissue_{sample}_by_{latent_cluster_key}.png')

#### 2.6.4 Compute EMD Distance

In [ ]:
# each list should at least contain 20 gene each 
# to have score that is meaningful
# compute mmd also and add it 
cell_gene_ensembl_id = list(top_gene_df['ensembl_id'])
neighborhood_gene_ensembl_id = list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)


In [ ]:
output_emd_distance = get_emd_distance( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
adata.obs['emd_dist'] = output_emd_distance

In [ ]:
# 2) Normalize your data so that 0→0; 0.6→1.0
norm = Normalize(vmin=0.0, vmax=max(output_emd_distance))

# 3) Build the colormap
#cmap = make_thresholded_cmap(base_cmap="plasma", threshold=0.8)

sq.pl.spatial_scatter(
        adata,
        color='emd_dist',
        #cmap=cmap,
        norm=norm,
        shape=None,
        figsize=(10, 8),
        size=1.5,
        title=f"{dataset_name} - (EMD distance)",
)
plt.show()

#### 2.6.5 Compute Spatial Score
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
adata.var

In [ ]:
# for out put of get_top_gene_score be correct both cell_gene_ensembl_id and 
# neighborhood_gene_ensembl_id should have same size and same order
cell_gene_ensembl_id =  list(top_gene_df['ensembl_id']) 
neighborhood_gene_ensembl_id =  list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)

In [ ]:
output_cosine_sim_score = get_spatial_score(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
gene_pair_score = output_cosine_sim_score['cos_sim_neighborhood'] / output_cosine_sim_score['cos_sim_cell']

In [ ]:
gene_pair_score

In [ ]:
output_cosine_sim_score['cell_count_neighborhood']

In [ ]:
df_gene_scores = get_top_gene_score(gene_pair_score, 
                                    cell_gene_ensembl_id, 
                                    gene_df=adata.var, 
                                    gene_counts= output_cosine_sim_score['cell_count_neighborhood'],
                                    min_count = 20)

In [ ]:
df_gene_scores.sort_values(by='gene_score', ascending=False)

In [ ]:
df_gene_pairs_score = get_top_gene_pairs(
    torch.from_numpy(gene_pair_score),
    count_cell=torch.from_numpy(output_cosine_sim_score['cell_count_neighborhood']),
    count_neb=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_cell=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_neb=torch.from_numpy(output_cosine_sim_score['cos_sim_neighborhood']),
    gene_df=adata.var,
    cell_gene_ids=cell_gene_ensembl_id,
    neighborhood_gene_ids=neighborhood_gene_ensembl_id,
    min_count=20,
    sim_thresh=0.0,
    k=500
)


In [ ]:
df_gene_pairs_score

### 2.7 Analyze Subsetted Data

Here we show how to subset your dataset to a specific set of cells using cell IDs (e.g. specific niche, cell type, ...). You can then repeat all the analysis from 2.6 on this subset of the data.

In [ ]:
subset_cell_ids = ['1002_batch40_0',
 '1002_batch40_1',
 '1002_batch40_2',
 '1002_batch40_3',
 '1002_batch40_4',
 '1002_batch40_5',
 '1002_batch40_6',
 '1002_batch40_7',
 '1002_batch40_8',
 '1002_batch40_9']

In [ ]:
dataset_subset = subset_by_cell_ids(dataset, subset_cell_ids)
dataset_subset

### 2.8 Perform Perturbations

#### 2.1.13 Perturbed Data

In [ ]:
# Define perturbations
perturbed_cell_id = ['1007_batch20_0', '1007_batch20_0', '1007_batch20_1']
perturbed_ensembl_id = ['all', 'all', 'ENSG00000169194'] # ['all'] * len(perturbed_cell_id)
perturbation_target = ['cell', 'neighborhood', 'cell'] # ['neighborhood'] * len(perturbed_cell_id)
perturbation_type = ['knockout', 'knockout', 'foldchange'] # ['knockout'] * len(perturbed_cell_id)
foldchange = [np.nan, np.nan, 0.5] # [np.nan] * len(perturbed_cell_id)

perturb_df =  pd.DataFrame({
    'perturbed_cell_id': perturbed_cell_id,
    'perturbed_ensembl_id': perturbed_ensembl_id,
    'perturbation_target': perturbation_target,
    'perturbation_type': perturbation_type, 
    'foldchange': foldchange
})

In [ ]:
perturbed_dataset = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    model_folder_path=model_folder_path)

#### 2.1.14 Embed Perturb Data

In [ ]:
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    cell_gene_ids=cell_gene_ids,
    neighborhood_gene_ids=neighborhood_gene_ids,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
adata.obsm['perturbed_cell_emb'] = perturbed_output_embed['cell_emb']
adata.obsm['perturbed_neighborhood_emb'] = perturbed_output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_neighborhood_emb',
                key_added='perturbed_neighborhood')
sc.tl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           key_added='perturbed_neighborhood_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_neighborhood_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_cell_emb',
                key_added='perturbed_cell')
sc.tl.umap(adata,
           neighbors_key='perturbed_cell',
           key_added='perturbed_cell_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_cell_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_cell',
           color=label)